# 1回目の試行

In [ ]:
import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# =======================================================
# ★設定エリア：ここを変えるだけで全て連動します★
# =======================================================
LAYER = 12                 # 見たいレイヤー (例: 12, 14, 20)
TEXT  = "私はステーキです。"  # 解析したいテキスト
# =======================================================

# 1. デバイス設定
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# 2. モデルのロード（Gemma-2-2B-JP）
# ※メモリ上に既に 'model' がある場合はロードをスキップする安全策を追加しても良いですが、
#   再現性確保のため、ここでは毎回確実に定義する構成にしています。
print("Loading Model...")
jp_model_name = "google/gemma-2-2b-jpn-it"

try:
    # Transformersで実体をロード
    hf_model = AutoModelForCausalLM.from_pretrained(
        jp_model_name,
        torch_dtype=torch.bfloat16,
        device_map=device
    )
    tokenizer = AutoTokenizer.from_pretrained(jp_model_name)

    # HookedTransformerでラップ（Gemma 2用エラー回避オプション付き）
    model = HookedTransformer.from_pretrained(
        "gemma-2-2b",
        hf_model=hf_model,
        device=device,
        tokenizer=tokenizer,
        dtype=torch.bfloat16,
        fold_ln=False,
        center_writing_weights=False,
        center_unembed=False,
        fold_value_biases=False,
    )
    print("✅ Model Loaded.")

except Exception as e:
    print(f"❌ Model Load Error: {e}")
    print("ヒント: HuggingFaceの認証トークン設定が済んでいない可能性があります。")
    raise e


# 3. SAE（レンズ）のロード
print(f"Loading SAE for Layer {LAYER}...")
release = "gemma-scope-2b-pt-res-canonical"
sae_id  = f"layer_{LAYER}/width_16k/canonical"
hook_name = f"blocks.{LAYER}.hook_resid_post"

try:
    sae = SAE.from_pretrained(
        release=release,
        sae_id=sae_id,
        device=device
    )
    print("✅ SAE Loaded.")
except Exception as e:
    print(f"❌ SAE Load Error: {e}")
    raise e


# 4. 実験実行（解析パート）
print(f"\nAnalyzing text: '{TEXT}'")

# キャッシュ取得
_, cache = model.run_with_cache(TEXT, names_filter=[hook_name])
input_activations = cache[hook_name]

# 特徴量抽出
feature_acts = sae.encode(input_activations)

# 結果表示
token_index = -1 
top_k = 5
specific_token_acts = feature_acts[0, token_index]
top_vals, top_inds = torch.topk(specific_token_acts, k=top_k)

target_token_str = model.to_string(model.to_tokens(TEXT)[0, token_index])

print(f"\n--- Result (Layer {LAYER}) ---")
print(f"Token looked at: '{target_token_str}'")
print("Top activated features:")

for score, feature_id in zip(top_vals, top_inds):
    if score > 0.1: 
        f_id = feature_id.item()
        print(f"Feature ID: {f_id} | Score: {score.item():.2f}")
        
        # URL自動生成
        url = f"https://www.neuronpedia.org/gemma-2-2b/{sae_id}/{f_id}"
        print(f"  -> {url}")
    else:
        print(f"(Feature ID: {feature_id.item()} is too weak)")

# AIに回答を出力させる。
output = model.generate(TEXT, max_new_tokens=100, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)

# 2回目以降

In [ ]:
import torch
from sae_lens import SAE

# =======================================================
# ★設定エリア：ここを変えるだけで全て連動します★
# =======================================================
LAYER = 13  # 見たいレイヤー番号 (例: 12, 14, 20)
TEXT  = "白露の色は一つをいかにして"
# =======================================================

# 1. 変数からIDと接続先を自動生成
release = "gemma-scope-2b-pt-res-canonical"
sae_id  = f"layer_{LAYER}/width_16k/canonical"
hook_name = f"blocks.{LAYER}.hook_resid_post"

print(f"Loading SAE for Layer {LAYER}...")
print(f" -> SAE ID: {sae_id}")
print(f" -> Hook  : {hook_name}")

# SAEロード
sae = SAE.from_pretrained(
    release=release,
    sae_id=sae_id,
    device=device
)
print("✅ SAE Loaded.")

# 2. 実験実行
print(f"\nAnalyzing text: '{TEXT}'")

# キャッシュ取得
_, cache = model.run_with_cache(TEXT, names_filter=[hook_name])
input_activations = cache[hook_name]

# 特徴量抽出
feature_acts = sae.encode(input_activations)

# 3. 結果表示
token_index = -1 
top_k = 5
specific_token_acts = feature_acts[0, token_index]
top_vals, top_inds = torch.topk(specific_token_acts, k=top_k)

target_token_str = model.to_string(model.to_tokens(TEXT)[0, token_index])

print(f"\n--- Result (Layer {LAYER}) ---")
print(f"Token looked at: '{target_token_str}'")
print("Top activated features:")

for score, feature_id in zip(top_vals, top_inds):
    if score > 0.1: 
        f_id = feature_id.item()
        print(f"Feature ID: {f_id} | Score: {score.item():.2f}")
        
        # Neuronpediaの短縮URL形式を使用 (例: 12-gemmascope-res-16k)
        # canonicalやl0値を使わず、レイヤー番号だけでリンクします
        url = f"https://www.neuronpedia.org/gemma-2-2b/{LAYER}-gemmascope-res-16k/{f_id}"
        print(f"  -> {url}")
    

# AIに回答を出力させる。
output = model.generate(TEXT, max_new_tokens=150, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)